# Music Genre Classification - clean

In [1]:
import datetime
import os
from pathlib import Path
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import librosa
import soundfile as sf

### Set constants, check for required dirs

In [2]:
# Confirm that DATA_ROOT dir is readable
DATA_ROOT = Path('../Data_Music')
print('DATA_ROOT dir exists:', DATA_ROOT.exists())

# Confirm that GENRES_DIR is readable
GENRES_DIR = DATA_ROOT / 'genres_original'
print('GENRES_DIR dir exists:', GENRES_DIR.exists())

# Print found genres
GENRES = sorted([d.name for d in GENRES_DIR.iterdir() if d.is_dir()])
print('Genres found:', GENRES)

DATA_ROOT dir exists: True
GENRES_DIR dir exists: True
Genres found: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


### Make dir for processed audio files

In [3]:
# Confirm that PROCESSED_DIR is readable
PROCESSED_DIR = DATA_ROOT / 'processed'
print('PROCESSED_DIR dir exists:', PROCESSED_DIR.exists())

for g in GENRES:
    subdir = PROCESSED_DIR / g
    if not os.path.exists(subdir):
        os.makedirs(subdir)

PROCESSED_DIR dir exists: True


### Define utils for finding audio files and checking metadata

In [4]:
def find_audio_files_within_dir(dir):
    # Traverse genre dirs, find names of each audio file
    found_audio_files = []
    for g in GENRES:
        for f in sorted((dir / g).glob('*.wav')):
            found_audio_files.append({'genre': g, 'filename': f.name, 'path': str(f)})

    # Convert found_audio_files to a pandas dataframe
    return pd.DataFrame(found_audio_files)

In [5]:
def get_metadata(audio_files_df):
    # Create empty list to store metadata of both readable and corrupted files
    metadata = []

    # Loop through every found audio file
    for audio_file in audio_files_df.itertuples(index=False):
        datum = {
            'genre': audio_file.genre,
            'filename': audio_file.filename,
            'path': audio_file.path,
        }
        try:
            # Read only the header of the file (fast, not full audio decode)
            info = sf.info(audio_file.path)
            datum.update({
                'duration': info.samplerate,
                'sample_rate': info.samplerate,
                'channels': info.channels,
                'corrupt': False,
            })
        except Exception as e:
            datum.update({
                'duration': np.nan,
                'sample_rate': np.nan,
                'channels': np.nan,
                'corrupt': True,
            })

        metadata.append(datum)

    # Convert metadata list to a dataframe
    return pd.DataFrame(metadata)

### Gather info about original audio files

We can see that there is only one corrupt audio file, which we know from `soundfile` throwing an exception when reading the header of the file. We see that all of our files have the same sample rate and roughly the same duration.

In [6]:
# Do the gathering
original_metadata_df = get_metadata(
    find_audio_files_within_dir(
        GENRES_DIR
    )
)

# Print helpful stats
readable_file_count = len(original_metadata_df[original_metadata_df['corrupt'] == False])
corrupt_file_count = len(original_metadata_df[original_metadata_df['corrupt'] == True])
print(f'Readable files: {readable_file_count}')
print(f'Corrupt files: {corrupt_file_count}')
print('---')
print('Sample rate values\n', original_metadata_df['sample_rate'].value_counts())
print('---')
print('Duration stats\n', original_metadata_df['duration'].value_counts())

Readable files: 999
Corrupted files: 1

Sample rate values:
sample_rate
22050    999
Name: count, dtype: int64

Duration stats:
count    999.000000
mean      30.024071
std        0.080951
min       29.931973
25%       30.000181
50%       30.013333
75%       30.013333
max       30.648889
Name: duration, dtype: float64


### Split audio into 3-second samples

Since ~1,000 data points isn't sufficient to make credible model, we split the audio files into three second samples. The resulting files are stored in the `Data_Music/processed` subdir. This operation doesn't take too long, just 18 seconds on a cheap, old Dell desktop.

In [ ]:
chunk_len_in_sec = 3
readable_audio_df = original_metadata_df[original_metadata_df['corrupt'] == False]

# Loop over metadata
for audio_file in readable_audio_df.itertuples():
    # Audio
    y, sr = librosa.load(Path(audio_file.path), mono='mono')

    # Goal: Carve the audio file into 3-sec chunks
    # Make dir for chunk files
    filename_without_ext = '.'.join(audio_file.filename.split('.')[0:-1])
    subdir = PROCESSED_DIR / audio_file.genre
    if not os.path.exists(subdir):
        os.makedirs(subdir)

    # First compute the start and end indices, then do the carving
    chunk_len_in_samples = chunk_len_in_sec * sr
    indicies = []
    for i in range(0, len(y), chunk_len_in_samples):
        end = i + chunk_len_in_samples
        if end <= len(y):
            indicies.append((i, end))

    for i, (start, end) in enumerate(indicies):
        path = subdir / f'{filename_without_ext}.{i:02}.wav'
        sf.write(path, y[start:end], sr)

### Interpret results from audio file splitting

The number of readable files might seem *odd*.

Originally, we had 999 readable, 30-sec files and one corrupt file, however we now we only have 9,981 3-sec files, instead of 9,990. The reason for this difference is that some 30-sec files were slightly shorter than 30-sec. Because we specified the splits to be exactly three seconds long, some 30-sec files produced only 9 samples.

This approach seemed appropriate to us because we wanted to ensure that all data inputted into our ML model had the same duration. With this approach, we reduceed our total dataset duration by 27 seconds (ie. (9,990 - 9,981) * 3), which seems insignificant to us. The upshot is we don't have to pad/truncate our audio files, which is quite nice!

To make sure that our dataset isn't too unbalanced, as a result of this split operation, let's revisit the class balance visualization from the prior EDA notebook.

In [8]:
metadata_df = get_metadata(
    find_audio_files_within_dir(
        PROCESSED_DIR
    )
)

# Print helpful stats
readable_file_count = len(original_metadata_df[original_metadata_df['corrupt'] == False])
corrupt_file_count = len(original_metadata_df[original_metadata_df['corrupt'] == True])
print(f'Readable files: {readable_file_count}')
print(f'Corrupt files: {corrupt_file_count}')
print('---')
print('Sample rate values\n', original_metadata_df['sample_rate'].value_counts())
print('---')
print('Duration stats\n', original_metadata_df['duration'].value_counts())

Readable files: 999
Corrupted files: 0

Sample rate values:
sample_rate
22050    999
Name: count, dtype: int64

Duration stats:
count    999.000000
mean      30.000061
std        0.000109
min       30.000000
25%       30.000000
50%       30.000000
75%       30.000181
max       30.000862
Name: duration, dtype: float64


Below we see that the classes aren't excessively unbalanced. Jazz is least represented genre, with only 990 samples, compared with several other genres that are fully represented, like blues and metal. Looks fine to continue on with the data processing!

In [ ]:
plt.figure(figsize=(10, 5))

counts = metadata_df['genre'].value_counts().sort_index()
ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index,
                 palette="viridis", legend=False)
ax.set_title('Class Distribution — Number of Audio Clips per Genre')
ax.set_xlabel('Genre')
ax.set_ylabel('Number of clips')
for i, v in enumerate(counts.values):
    ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Extract features from audio data using librosa

The features of interest are:
* tempo
* mean chroma
* mfcc means (with n_mfcc=13)
* spectral centroid mean
* pectral rolloff mean
* zero crossing rate mean

This operation 4.5 min to run on that previously mentioned under-powered Dell desktop.

In [14]:
# Setup time recording
print('Begin extracting features')
print('This will take a few minutes')
print('Start time:', datetime.datetime.now())
start = time.time()

mono = True
extracted_features_list = []

for path in find_audio_files_within_dir(PROCESSED_DIR)['path'].to_list():
    # Audio
    y, sr = librosa.load(Path(path), mono=mono)

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    
    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma)
    
    # MFCC (13 coefficients)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    
    #Normalize the MFCC array by computing the mean to 0 and computing the standard deviation to 1 
    mfcc_normalized = (mfcc - np.mean(mfcc, axis=1, keepdims=True)) / (np.std(mfcc, axis=1, keepdims=True) + 1e-8)
    
    # Spectral features
    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    extracted_features_list.append({
        'tempo': tempo[0],
        'chroma_mean': chroma_mean,
        'mfcc1_normalized': mfcc_normalized[0],
        'mfcc2_normalized': mfcc_normalized[1],
        'mfcc3_normalized': mfcc_normalized[2],
        'mfcc4_normalized': mfcc_normalized[3],
        'mfcc5_normalized': mfcc_normalized[4],
        'mfcc6_normalized': mfcc_normalized[5],
        'mfcc7_normalized': mfcc_normalized[6],
        'mfcc8_normalized': mfcc_normalized[7],
        'mfcc9_normalized': mfcc_normalized[8],
        'mfcc10_normalized': mfcc_normalized[9],
        'mfcc11_normalized': mfcc_normalized[10],
        'mfcc12_normalized': mfcc_normalized[11],
        'mfcc13_normalized': mfcc_normalized[12],
        'spectral_centroid': spectral_centroid,
        'spectral_rolloff': spectral_rolloff,
        'zero_crossing_rate': zcr
    })

extracted_features_df = pd.DataFrame(extracted_features_list)
display(extracted_features_df)

# Print runtime
print('End time:', datetime.datetime.now())
elapsed_time = time.time() - start
print(f'Elapsed time {np.floor(elapsed_time / 60):.0f} min, {round(elapsed_time % 60)} sec')
print()
print('Extracted features below:')

Begin extracting features
This will take a few minutes
Start time: 2026-07-03 23:33:28.246420


,tempo,chroma_mean,mfcc1_normalized,mfcc2_normalized,mfcc3_normalized,mfcc4_normalized,mfcc5_normalized,mfcc6_normalized,mfcc7_normalized,mfcc8_normalized,mfcc9_normalized,mfcc10_normalized,mfcc11_normalized,mfcc12_normalized,mfcc13_normalized,spectral_centroid,spectral_rolloff,zero_crossing_rate
0,123.046875,0.349951,"[-2.50509, -1.9248292, -1.58352, -1.2030671, -...","[-1.2738171, -1.1926856, -1.1228391, -0.845542...","[0.765397, 0.70316005, 1.372534, 1.0617235, 0....","[-0.39655474, 0.2838523, 0.92198575, 1.0440992...","[0.8702419, 0.788286, 0.7301766, 1.1902823, 0....","[0.048039794, 0.24011798, 0.90292037, 1.138586...","[-0.41866425, -0.13148117, -0.13486952, -0.499...","[-1.8409187, -1.6485567, -0.65470284, -0.76248...","[0.42829868, 0.24192335, 0.64226776, 1.1864132...","[0.42673758, 0.66121143, 0.9044465, 0.8540269,...","[0.7751633, 0.5375513, -0.10971849, -0.9517254...","[-1.78547, -1.0800276, -1.5267029, -1.706918, ...","[0.42089054, 0.35488018, -0.027885843, 0.18296...",1784.416546,3806.418650,0.083066
1,67.999589,0.340945,"[-1.4924778, -0.18511394, 0.58440226, 0.665564...","[1.4247956, 0.8803709, 0.42164722, 0.05543874,...","[0.11760394, 0.16937488, 0.12503065, 0.3469864...","[-1.181175, 0.4440553, 0.7606207, 1.2130176, 1...","[0.948397, -0.219803, -0.43929914, -0.34182084...","[-1.3372147, 0.9892304, 1.8379145, 1.576901, 0...","[1.2128806, 0.88929516, 0.67691255, 0.09766668...","[-2.0208404, -0.8400674, -0.13592567, 0.064594...","[-0.16723102, 0.4297649, 0.6369697, 0.72553366...","[-1.701058, -1.2932531, 0.008401172, 0.2651074...","[0.52249014, 0.6553496, 0.35096812, 0.08417034...","[-1.4256743, -0.2034508, 0.31971312, 0.7296446...","[-0.3512019, -0.2947848, 0.032164, 0.4938203, ...",1529.871314,3548.986873,0.056044
2,161.499023,0.363562,"[-0.035139814, 0.22629374, -0.0973047, -0.3091...","[0.29949096, 0.2719848, 0.05779178, 0.00477574...","[-1.1727895, -0.67054373, -0.17092188, -0.0866...","[-1.4941461, -1.0184939, -0.61025906, -0.53751...","[0.9960475, 0.99013203, 0.7129133, 0.5745672, ...","[0.9553223, 0.98244256, 0.8962313, 0.93065995,...","[0.87775445, 0.65922314, 0.70613617, 0.0261021...","[-0.10677589, -0.011422865, -0.16311894, -0.46...","[-0.37631688, -0.36339378, 0.025445657, -0.368...","[-0.52243644, 0.17829317, 0.5706336, -0.031054...","[-0.79830134, -0.29204205, 0.5841326, 0.895188...","[0.26771516, 0.33603418, 0.554314, 0.7539845, ...","[0.05983896, -0.96374, -0.951537, -0.029783849...",1552.637786,3041.089944,0.076301
3,63.024009,0.404848,"[-0.28589296, -0.027815992, -0.18817371, -0.28...","[1.4309267, 1.3146316, 1.3119252, 1.4593377, 1...","[-0.47427934, -0.50694126, -0.46592245, -0.728...","[-0.5513107, -0.07175644, 0.078941554, -0.1442...","[0.47238365, 0.33310166, 0.40498766, 0.6000734...","[-0.60711926, -0.29072896, -0.5930833, -0.9882...","[0.6229035, 0.8614692, 1.0877017, 0.95214117, ...","[0.6911793, 0.9190839, 0.9466656, 0.9892487, 0...","[2.1922882, 1.6848142, 1.5510391, 2.0867364, 1...","[0.14136693, 0.06437835, -0.23640744, 0.179063...","[0.81645685, 0.516102, 0.61279017, 1.1552601, ...","[0.08941053, 0.42221624, 0.3371246, -0.0852567...","[0.8880079, 0.85974044, 0.787575, 0.37382355, ...",1070.110059,2185.061787,0.033309
4,135.999178,0.308598,"[-1.6850829, -0.27805567, 0.073142044, -0.1786...","[0.5812951, 0.44695893, 0.19292042, 0.31263322...","[0.47467852, -0.5703597, -0.91013455, -0.75702...","[-1.8881474, -1.0531039, -0.56576484, -0.29775...","[0.8593501, -0.30539325, -0.47553122, -0.04250...","[-1.1596519, -0.11731192, -0.10196566, -0.6336...","[0.89820427, 0.33792838, -0.020404322, 0.13869...","[-0.41032642, -0.3163635, -0.7295778, -1.17515...","[0.82330006, 0.020842426, -0.3627909, -0.39646...","[-1.5525242, -1.0923848, -1.2254702, -1.131820...","[-0.77347803, -0.5656913, -0.5601509, -0.15165...","[-1.1194482, -0.8874215, -0.32261994, -0.09369...","[0.9531465, 2.3007305, 1.9012917, 1.5212953, 1...",1835.507008,3581.003346,0.101500
...,...,...,...,...,...,...,...,...,...,...,...,.

End time: 2026-07-03 23:36:52.439031
Elapsed time 3 min, 24 sec


In [11]:
# Combine metadata and extracted features into one DataFrame
full_df = pd.concat([metadata_df, extracted_features_df], axis=1)
full_df

,genre,filename,path,duration,sample_rate,channels,tempo,chroma_mean,mfcc1_mean,mfcc2_mean,...,mfcc7_mean,mfcc8_mean,mfcc9_mean,mfcc10_mean,mfcc11_mean,mfcc12_mean,mfcc13_mean,spectral_centroid,spectral_rolloff,zero_crossing_rate
0,blues,blues.00000.wav,../Data_Music/processed/blues/blues.00000.wav,30.0,22050,1,123.046875,0.349951,-113.619385,121.553017,...,-13.692060,15.339378,-12.283617,10.973775,-8.322410,8.806787,-3.665802,1784.416546,3806.418650,0.083066
1,blues,blues.00001.wav,../Data_Music/processed/blues/blues.00001.wav,30.0,22050,1,67.999589,0.340945,-207.581512,123.997147,...,-8.555368,23.355938,-10.101037,11.906445,-5.558123,5.375942,-2.237833,1529.871314,3548.986873,0.056044
2,blues,blues.00002.wav,../Data_Music/processed/blues/blues.00002.wav,30.0,22050,1,161.499023,0.363562,-90.776344,140.448608,...,-13.644712,11.623112,-11.775921,9.700466,-13.115349,5.785763,-8.899733,1552.637786,3041.089944,0.076301
3,blues,blues.00003.wav,../Data_Music/processed/blues/blues.00003.wav,30.0,22050,1,63.024009,0.404848,-199.462006,150.094711,...,-4.828873,9.297849,-0.753142,8.147393,-3.195236,6.085353,-2.476188,1070.110059,2185.061787,0.033309
4,blues,blues.00004.wav,../Data_Music/processed/blues/blues.00004.wav,30.0,22050,1,135.999178,0.308598,-160.291855,126.195770,...,-23.357162,0.500523,-11.804770,1.203877,-13.085074,-2.809849,-6.935621,1835.507008,3581.003346,0.101500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,rock,rock.00095.wav,../Data_Music/processed/rock/rock.00095.wav,30.0,22050,1,103.359375,0.351992,-153.616486,109.894508,...,-16.486076,18.083521,-22.215626,15.403847,-12.378472,12.349251,-12.316458,2008.537046,4254.124276,0.089267
995,rock,rock.00096.wav,../Data_Music/processed/rock/rock.00096.wav,30.0,22050,1,117.453835,0.398761,-142.442062,116.238441,...,-18.311134,20.106487,-22.106911,10.796755,-13.001553,14.067452,-16.397627,2006.009248,4147.166589,0.097659
996,rock,rock.00097.wav,../Data_Music/processed/rock/rock.00097.wav,30.0,22050,1,129.199219,0.431909,-125.065109,115.203308,...,-12.428562,20.139965,-18.370886,10.119690,-16.055443,10.486740,-17.959637,2077.166788,4030.750627,0.121824
997,rock,rock.00098.wav,../Data_Music/processed/rock/rock.00098.wav,30.0,22050,1,73.828125,0.362428,-224.994049,123.669685,...,-10.257257,15.573306,-8.251973,12.805178,-9.004445,7.690067,-10.085666,1398.581574,3014.673437,0.048731


### Store the fully processed data

Here we save the full data DataFrame as a **csv file**, and put it in the processed subdir of the `Data_Music` folder. This csv file is 3 MiB in size and can be used by the SQL notebook. As a reminder, the SQL notebook will ensure that the data we have just written to disk is stored in a reliable and queryable way. Once it's in that state, it will be used by our model notebook.

In [12]:
full_df.to_csv(PROCESSED_DIR / 'data.csv', index=False)